In [96]:
from io import StringIO
import time
import os
import requests
import pandas as pd
import numpy as npU
import yfinance as yf

---

# Exploratorie

In [3]:
# URL da página com a lista de empresas do S&P 500
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Definindo o cabeçalho para a requisição HTTP
headers = {'User-Agent': 'Mozilla/5.0'}

In [4]:
# Fazendo a requisição HTTP para obter o conteúdo da página
response = requests.get(url, headers=headers)

# Lendo as tabelas da página HTML usando pandas
tables = pd.read_html(StringIO(response.text))

In [5]:
# Exibindo o número de tabelas encontradas
print(len(tables))   

# Exibindo as primeiras linhas da primeira tabela
tables[0].head()

2


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [6]:
tables[1].head(3)

,vteS&P 500 companies,vteS&P 500 companies.1
0,Energy,APA Corporation Baker Hughes Chevron Corporati...
1,Materials,Air Products Albemarle Corporation Amcor Avery...
2,Industrials,3M A. O. Smith Allegion Ametek Automatic Data ...


In [10]:
tables[0]['Security'].unique()

<StringArray>
[                    '3M',            'A. O. Smith',    'Abbott Laboratories',
                 'AbbVie',              'Accenture',             'Adobe Inc.',
 'Advanced Micro Devices',        'AES Corporation',                  'Aflac',
   'Agilent Technologies',
 ...
     'Williams Companies',   'Willis Towers Watson',          'Workday, Inc.',
           'Wynn Resorts',            'Xcel Energy',             'Xylem Inc.',
            'Yum! Brands',     'Zebra Technologies',          'Zimmer Biomet',
                 'Zoetis']
Length: 503, dtype: str

In [11]:
tables[0]['Symbol'].unique()

<StringArray>
[ 'MMM',  'AOS',  'ABT', 'ABBV',  'ACN', 'ADBE',  'AMD',  'AES',  'AFL',
    'A',
 ...
  'WMB',  'WTW', 'WDAY', 'WYNN',  'XEL',  'XYL',  'YUM', 'ZBRA',  'ZBH',
  'ZTS']
Length: 503, dtype: str

In [13]:
print(type(tables))

<class 'list'>


In [14]:
# Se for uma lista, verifica o tipo do primeiro item
if isinstance(tables, list) and len(tables) > 0:
    print(type(tables[0]))

<class 'pandas.DataFrame'>


In [15]:
# Transformando as tabelas em DataFrames do pandas
df_constituents = tables[0]
df_changes = tables[1]

In [ ]:
df_constituents.head(3)

In [ ]:
df_changes.head(3)

In [12]:
df_constituents.info()

<class 'pandas.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Symbol                 503 non-null    str  
 1   Security               503 non-null    str  
 2   GICS Sector            503 non-null    str  
 3   GICS Sub-Industry      503 non-null    str  
 4   Headquarters Location  503 non-null    str  
 5   Date added             503 non-null    str  
 6   CIK                    503 non-null    int64
 7   Founded                503 non-null    str  
dtypes: int64(1), str(7)
memory usage: 31.6 KB


In [13]:
# Convertendo a coluna 'Date added' para datetime
df_constituents['Date added'] = pd.to_datetime(df_constituents['Date added'], errors='coerce')

In [14]:
# Adicionando uma nova coluna com o ano de adição
df_constituents['Year added'] = df_constituents['Date added'].dt.year

---

# Functions

In [ ]:
# Função para fazer scraping dos dados do S&P 500
def scrapping_snp500(n, urls):
    # URL da página com a lista de empresas do S&P 500
    url = urls

    # Definindo o cabeçalho para a requisição HTTP
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    # Fazendo a requisição HTTP para obter o conteúdo da página
    response = requests.get(url, headers=headers)

    # Lendo as tabelas da página HTML usando pandas
    tables = pd.read_html(StringIO(response.text))
    df = tables[n]  # A primeira tabela contém a lista de empresas do S&P 500
    
    # Verificando se a tabela contém a coluna 'Date added'
    if n == 0:
        # Convertendo a coluna 'Date added' para datetime
        df['Date added'] = pd.to_datetime(df['Date added'], errors='coerce')
        # Adicionando uma nova coluna com o ano de adição
        df['Year added'] = df['Date added'].dt.year
    else:
        return df
        
    return df

In [3]:
# Função para salvar um DataFrame em um arquivo CSV
def save_dataframe(df, file_name):
    base_dir = r"C:\Users\leona\github\stock-markets-analytics-zoomcamp-2026\data"
    file_path = os.path.join(base_dir, f"{file_name}.csv")
    df.to_csv(file_path, index=False)


In [129]:
# Função para obter dados históricos atraves de tickers usando yfinance
def get_historical_data_org(df, start_date, end_date):
    """
    Função para obter dados históricos atraves de tickers usando yfinance.
    
    Parâmetros:
    - df: DataFrame contendo os símbolos das ações (string)
    - start_date: Data de início no formato 'YYYY-MM-DD' (string)
    - end_date: Data de término no formato 'YYYY-MM-DD' (string)
    
    Retorna:
    - DataFrame com os dados históricos
    """
    tickers_list = df['Ticker'].tolist()
    result = {}
    for s in tickers_list:
        result[s] = yf.download(s, start=start_date, end=end_date, multi_level_index=False)
        
    df = pd.concat(result, names=['Ticker', 'Date']).reset_index()
    return df

In [100]:
def get_historical_data(df, start_date, end_date, batch_size=50, sleep_seconds=2):
    """
    Function to get historical data for a list of tickers using yfinance.

    Parameters:
    - df: DataFrame containing the ticker symbols (must have a 'Ticker' column)
    - start_date: Start date in 'YYYY-MM-DD' format (string)
    - end_date: End date in 'YYYY-MM-DD' format (string)
    - batch_size: number of tickers to download before pausing (default 50)
    - sleep_seconds: pause duration in seconds between batches (default 2)

    Returns:
    - df_result: DataFrame with the historical data
    - failed_tickers: list of tickers that failed to download
    """
    tickers_list = df['Ticker'].tolist()
    result = {}
    failed_tickers = []

    for i, ticker in enumerate(tickers_list):
        try:
            data = yf.download(ticker, start=start_date, end=end_date, multi_level_index=False)
            if data.empty:
                failed_tickers.append(ticker)  # download "worked" but came back empty
            else:
                result[ticker] = data
        except Exception:
            failed_tickers.append(ticker)  # download raised an error

        if (i + 1) % batch_size == 0:
            time.sleep(sleep_seconds)  # pause every `batch_size` tickers

    df_result = pd.concat(result, names=['Ticker', 'Date']).reset_index()
    return df_result, failed_tickers

In [121]:
def get_eu_historical_data(df, start_date, end_date, batch_size=50, sleep_seconds=2):
    """
    Function to get historical data for European tickers using yfinance.

    Handles the EU-specific issue where a ticker is missing its exchange
    suffix (e.g. 'ZURN' instead of 'ZURN.SW'): when a download fails or
    comes back empty AND the ticker has no known suffix, it looks up the
    ticker's Country and retries once with the corrected ticker.

    Parameters:
    - df: DataFrame with 'Ticker' and 'Country' columns
    - start_date, end_date: 'YYYY-MM-DD' strings
    - batch_size, sleep_seconds: pacing controls

    Returns:
    - df_result: DataFrame with the historical data
    - report: dict with four lists -
        'fixed'                     -> [(original, corrected), ...] suffix issues auto-fixed
        'missing_suffix_unresolved' -> no suffix AND unknown country (couldn't fix)
        'empty'                     -> already had a suffix, but returned no data
        'error'                     -> [(ticker, error_message), ...] already had a suffix, but raised an exception
    """
    country_suffix_map = {
        'Germany': '.DE', 'France': '.PA', 'Netherlands': '.AS', 'Switzerland': '.SW',
        'United Kingdom': '.L', 'Italy': '.MI', 'Spain': '.MC', 'Sweden': '.ST',
        'Norway': '.OL', 'Denmark': '.CO', 'Finland': '.HE', 'Belgium': '.BR',
        'Austria': '.VI', 'Portugal': '.LS', 'Ireland': '.IR',
    }

    tickers_list = df['Ticker'].tolist()
    country_lookup = df.set_index('Ticker')['Country']

    result = {}
    report = {'fixed': [], 'missing_suffix_unresolved': [], 'empty': [], 'error': []}

    for i, ticker in enumerate(tickers_list):
        # --- attempt 1: ticker as-is ---
        try:
            data = yf.download(ticker, start=start_date, end=end_date, multi_level_index=False)
            error = None if not data.empty else 'empty'
        except Exception as e:
            data, error = None, str(e)

        if error is None:
            result[ticker] = data

        else:
            has_suffix = any(ticker.endswith(suf) for suf in country_suffix_map.values())

            if not has_suffix:
                # --- likely a missing-suffix problem: build the corrected ticker and retry ---
                country = country_lookup.get(ticker)
                suffix = country_suffix_map.get(country) if country is not None else None

                if suffix is None:
                    report['missing_suffix_unresolved'].append(ticker)
                else:
                    fixed_ticker = ticker.replace('.', '-') + suffix  # 'BT.A' -> 'BT-A.L'

                    try:
                        data2 = yf.download(fixed_ticker, start=start_date, end=end_date, multi_level_index=False)
                        if not data2.empty:
                            result[fixed_ticker] = data2
                            report['fixed'].append((ticker, fixed_ticker))
                        else:
                            report['empty'].append(fixed_ticker)
                    except Exception as e2:
                        report['error'].append((fixed_ticker, str(e2)))

            else:
                # already had a suffix -> the failure is NOT about the suffix
                if error == 'empty':
                    report['empty'].append(ticker)
                else:
                    report['error'].append((ticker, error))

        if (i + 1) % batch_size == 0:
            time.sleep(sleep_seconds)

    df_result = pd.concat(result, names=['Ticker', 'Date']).reset_index()
    return df_result, report

In [5]:
# Função para calcular o crescimento ano a ano (YTD) para cada ticker
def get_prices(df, date):
    
    df = df[df['Date'] > date]
    # Pivot the DataFrame to have dates as index and tickers as columns
    close_prices = df.pivot(index= 'Date', columns='Ticker', values='Close')
    # Calculating the first and last prices for each ticker
    first_price = close_prices.bfill().iloc[0]
    last_price = close_prices.ffill().iloc[-1]
    # Calculating the year-to-date growth for each ticker
    growing = (last_price / first_price - 1) * 100
    return growing

In [6]:
# Function to compare the growth of each ticker against the S&P 500
def compare_growth(df):
    index_max_growth = df > df.loc['^GSPC']
    return index_max_growth

In [79]:
# Function to get the names of all tables in a given URL
def scrapping(urls, table):
    # URL da página com a lista de empresas
    url = urls

    # Definindo o cabeçalho para a requisição HTTP
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    # Fazendo a requisição HTTP para obter o conteúdo da página
    response = requests.get(url, headers=headers)
    
    # Verificando se a tabela está presente no conteúdo da página
    check_response = table in response.text
    
    # Lendo as tabelas da página HTML usando pandas
    if check_response == True:
        print(f'Table {table} status: {check_response}')
        # Lendo as tabelas da página HTML usando pandas
        tables = pd.read_html(StringIO(response.text), match=table)
        df = tables[0]  
    
    return df

In [80]:
# Função para obter os nomes das tabelas de uma página HTML
def get_html_tables_name(urls):
    """
    Função para obter os nomes das tabelas de uma página HTML.
    
    Parâmetros:
    - response: Objeto de resposta da requisição HTTP contendo o conteúdo da página.
    
    Retorna:
    - Lista com os nomes das tabelas encontradas na página.
    """
    
    # URL da página com a lista de empresas
    url = urls

    # Definindo o cabeçalho para a requisição HTTP
    headers = {'User-Agent': 'Mozilla/5.0'}

    # Fazendo a requisição HTTP para obter o conteúdo da página
    response = requests.get(url, headers=headers)
    
    # Lendo as tabelas da página HTML usando pandas
    tables = pd.read_html(StringIO(response.text))
    print(len(tables))

    # Exibindo o número de tabelas encontradas e as primeiras 5 colunas de cada tabela
    for i, t in enumerate(tables):
        print(i, t.shape, list(t.columns)[:5])

---

# Questions

### Question 1

In [31]:
df_constituents = scrapping(0)
df_constituents.head(3)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,Year added
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,1957
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,2017
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,1957


In [32]:
# Contando o número de ações adicionadas por ano
df_added_per_year = df_constituents.groupby('Year added').size().reset_index(name='Count')
df_added_per_year.head(3)

,Year added,Count
0,1957,52
1,1964,1
2,1965,2


In [33]:
# Encontrando o ano com o maior número de ações adicionadas
df_2020_plus = df_added_per_year[df_added_per_year['Year added'] >= 2020]
df_2020_plus.head(3)

,Year added,Count
51,2020,10
52,2021,10
53,2022,15


In [34]:
# Encontrando o ano com o maior número de ações adicionadas desde 2020
year_highest_stokes = df_2020_plus.loc[df_2020_plus['Count'].idxmax()]
year_highest_stokes

Year added    2025
Count           18
Name: 56, dtype: int64

In [35]:
# Contando o número de ações que estão na lista há mais de 20 anos
df_over_20_years = df_constituents[df_constituents['Date added'] <= '2006-01-01'].count()
df_over_20_years

Symbol                   218
Security                 218
GICS Sector              218
GICS Sub-Industry        218
Headquarters Location    218
Date added               218
CIK                      218
Founded                  218
Year added               218
dtype: int64

---

### Question 2

In [36]:
# 10 Index Tickers
index_10 = ['^GSPC', '000001.SS', '^HSI',
                '^AXJO', '^NSEI', '^GSPTSE',
                '^GDAXI', '^FTSE', '^N225',
                '^MXX', '^BVSP']

In [ ]:
# Obtendo dados históricos do S&P 500 de 1950 até 2026
df_index_10 = get_historical_data(index_10, '1950-01-01', '2026-08-21')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [38]:
# Calculando o crescimento para diferentes períodos
df_prices_2026 = get_prices(df_index_10, '2026-01-01')
df_prices_2023 = get_prices(df_index_10, '2023-01-01')
df_prices_2021 = get_prices(df_index_10, '2021-01-01')
df_prices_2016 = get_prices(df_index_10, '2016-01-01')


In [39]:
# Comparando o crescimento dos tickers em relação ao S&P 500
compare_ytd = compare_growth(df_prices_2026)
compare_ytd

Ticker
000001.SS    False
^AXJO        False
^BVSP        False
^FTSE        False
^GDAXI       False
^GSPC        False
^GSPTSE       True
^HSI         False
^MXX         False
^N225         True
^NSEI        False
dtype: bool

In [40]:
# Comparando o crescimento dos tickers em relação ao S&P 500
compare_10_years = compare_growth(df_prices_2016)
compare_10_years

Ticker
000001.SS    False
^AXJO        False
^BVSP         True
^FTSE        False
^GDAXI       False
^GSPC        False
^GSPTSE      False
^HSI         False
^MXX         False
^N225        False
^NSEI        False
dtype: bool

In [41]:
# Comparando o crescimento dos tickers em relação ao S&P 500
compare_5_years = compare_growth(df_prices_2021)
compare_5_years

Ticker
000001.SS    False
^AXJO        False
^BVSP        False
^FTSE        False
^GDAXI       False
^GSPC        False
^GSPTSE       True
^HSI         False
^MXX         False
^N225         True
^NSEI        False
dtype: bool

In [42]:
# Comparando o crescimento dos tickers em relação ao S&P 500
compare_3_years = compare_growth(df_prices_2023)
compare_3_years

Ticker
000001.SS    False
^AXJO        False
^BVSP        False
^FTSE        False
^GDAXI       False
^GSPC        False
^GSPTSE      False
^HSI         False
^MXX         False
^N225         True
^NSEI        False
dtype: bool

In [43]:
# Summarizing the results
print("YTD:", compare_ytd.sum())
print("10 anos:", compare_10_years.sum())
print("5 anos:", compare_5_years.sum())
print("3 anos:", compare_3_years.sum())

YTD: 2
10 anos: 1
5 anos: 2
3 anos: 1


---

### Question 3

In [44]:
# Obtendo dados históricos do S&P 500
df_historical_sp500 = df_index_10[df_index_10['Ticker'] == '^GSPC']
df_historical_sp500 = df_historical_sp500[['Date', 'Close']].reset_index(drop=True)
df_historical_sp500

,Date,Close
0,1950-01-03,16.660000
1,1950-01-04,16.850000
2,1950-01-05,16.930000
3,1950-01-06,16.980000
4,1950-01-09,17.080000
...,...,...
19275,2026-08-14,7785.759766
19276,2026-08-17,7745.060059
19277,2026-08-18,7691.759766
19278,2026-08-19,7707.979980


In [45]:
# Calculando o valor máximo acumulado do S&P 500
df_historical_sp500['cummax'] = df_historical_sp500['Close'].cummax()

In [46]:
# Identificando se o preço de fechamento é um novo recorde histórico
df_historical_sp500['is_ath'] = df_historical_sp500['Close'] == df_historical_sp500['cummax']

In [47]:
# Obtendo os dias em que o S&P 500 atingiu um novo recorde histórico
historical_max_days = df_historical_sp500[df_historical_sp500['is_ath']]
historical_max_days[['Date', 'Close']]

,Date,Close
0,1950-01-03,16.660000
1,1950-01-04,16.850000
2,1950-01-05,16.930000
3,1950-01-06,16.980000
4,1950-01-09,17.080000
...,...,...
19223,2026-06-01,7599.959961
19224,2026-06-02,7609.779785
19267,2026-08-04,7736.520020
19270,2026-08-07,7757.640137


In [48]:
historical_max_days = historical_max_days.sort_values('Date').reset_index(drop=True)

results = []

for i in range(len(historical_max_days) - 1):
    peak_date = historical_max_days.loc[i, 'Date']
    peak_price = historical_max_days.loc[i, 'Close']
    next_peak_date = historical_max_days.loc[i + 1, 'Date']

    # get only the days BETWEEN the two peaks (excluding the peak days themselves)
    period = df_historical_sp500[
        (df_historical_sp500['Date'] > peak_date) &
        (df_historical_sp500['Date'] < next_peak_date)
    ]

    if period.empty:
        continue  # the two peaks are consecutive days, no drop in between

    min_price = period['Close'].min()
    min_date = period.loc[period['Close'].idxmin(), 'Date']

    results.append({
        'peak_date': peak_date,
        'peak_price': peak_price,
        'min_date': min_date,
        'min_price': min_price,
        'next_peak_date': next_peak_date
    })

df_corrections = pd.DataFrame(results)
df_corrections.head()

,peak_date,peak_price,min_date,min_price,next_peak_date
0,1950-01-09,17.08,1950-01-10,17.030001,1950-01-11
1,1950-01-11,17.09,1950-01-13,16.670000,1950-02-02
2,1950-02-06,17.32,1950-02-16,16.990000,1950-03-06
3,1950-03-06,17.32,1950-03-09,17.070000,1950-03-15
4,1950-03-16,17.49,1950-03-20,17.440001,1950-03-22


In [49]:
# Calculate the drawdown percentage
df_corrections['drawdown_pct'] = (df_corrections['peak_price'] - df_corrections['min_price']) / df_corrections['peak_price'] * 100

In [50]:
# Filter the DataFrame to include only rows where the drawdown percentage is greater than or equal to 5%
df_significant = df_corrections[df_corrections['drawdown_pct'] >= 5]

In [51]:
# Calculate the duration of each drawdown in days
df_significant['duration_days'] = (df_significant['min_date'] - df_significant['peak_date']).dt.days

In [52]:
# Calculate percentiles for duration and drawdown
duration_percentiles = df_significant['duration_days'].quantile([0.25, 0.5, 0.75])
drawdown_percentiles = df_significant['drawdown_pct'].quantile([0.25, 0.5, 0.75])

print(duration_percentiles)
print(drawdown_percentiles)

0.25    22.00
0.50    40.50
0.75    86.25
Name: duration_days, dtype: float64
0.25     6.234677
0.50     7.986358
0.75    14.019826
Name: drawdown_pct, dtype: float64


---

### Question 4

In [ ]:
df_amzn = get_historical_data(['AMZN'], '2020-10-27', '2026-10-29')

[*********************100%***********************]  1 of 1 completed


In [108]:
df_amzn

,Ticker,Date,Close,High,Low,Open,Volume
0,AMZN,2020-10-27,164.316498,164.582993,160.565002,161.246994,85820000
1,AMZN,2020-10-28,158.139008,163.201004,158.123505,162.464996,111766000
2,AMZN,2020-10-29,160.550507,162.862503,158.199997,160.063507,131930000
3,AMZN,2020-10-30,151.807495,158.350006,150.949997,157.887497,167728000
4,AMZN,2020-11-02,150.223999,153.985504,147.505997,153.087006,145148000
...,...,...,...,...,...,...,...
1468,AMZN,2026-09-02,254.979996,256.239990,253.399994,254.259995,24013300
1469,AMZN,2026-09-03,258.899994,259.500000,256.000000,256.200012,26891500
1470,AMZN,2026-09-04,258.510010,261.119995,255.289993,259.070007,30749900
1471,AMZN,2026-09-08,256.970001,257.989990,254.750000,256.679993,29349000


In [109]:
# Calculando o preço de fechamento do dia anterior para cada linha do DataFrame
df_amzn['Close_Day1'] = df_amzn['Close'].shift(1)

In [110]:
# Calculando o preço de fechamento de dois dias atrás para cada linha do DataFrame
df_amzn['Close_Day3'] = df_amzn['Close'].shift(-1)

In [111]:
# Calculando o retorno de dois dias para cada linha do DataFrame
df_amzn['return_2d'] = df_amzn['Close_Day3'] / df_amzn['Close_Day1'] - 1

In [112]:
df_amzn

,Ticker,Date,Close,High,Low,Open,Volume,Close_Day1,Close_Day3,return_2d
0,AMZN,2020-10-27,164.316498,164.582993,160.565002,161.246994,85820000,NaN,158.139008,NaN
1,AMZN,2020-10-28,158.139008,163.201004,158.123505,162.464996,111766000,164.316498,160.550507,-0.022919
2,AMZN,2020-10-29,160.550507,162.862503,158.199997,160.063507,131930000,158.139008,151.807495,-0.040038
3,AMZN,2020-10-30,151.807495,158.350006,150.949997,157.887497,167728000,160.550507,150.223999,-0.064319
4,AMZN,2020-11-02,150.223999,153.985504,147.505997,153.087006,145148000,151.807495,152.420502,0.004038
...,...,...,...,...,...,...,...,...,...,...
1468,AMZN,2026-09-02,254.979996,256.239990,253.399994,254.259995,24013300,254.919998,258.899994,0.015613
1469,AMZN,2026-09-03,258.899994,259.500000,256.000000,256.200012,26891500,254.979996,258.510010,0.013844
1470,AMZN,2026-09-04,258.510010,261.119995,255.289993,259.070007,30749900,258.899994,256.970001,-0.007455
1471,AMZN,2026-09-08,256.970001,257.989990,254.750000,256.679993,29349000,258.510010,251.799896,-0.025957


In [113]:
df_amzn.info()

<class 'pandas.DataFrame'>
RangeIndex: 1473 entries, 0 to 1472
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype        
---  ------      --------------  -----        
 0   Ticker      1473 non-null   str          
 1   Date        1473 non-null   datetime64[s]
 2   Close       1473 non-null   float64      
 3   High        1473 non-null   float64      
 4   Low         1473 non-null   float64      
 5   Open        1473 non-null   float64      
 6   Volume      1473 non-null   int64        
 7   Close_Day1  1472 non-null   float64      
 8   Close_Day3  1472 non-null   float64      
 9   return_2d   1471 non-null   float64      
dtypes: datetime64[s](1), float64(7), int64(1), str(1)
memory usage: 115.2 KB


In [114]:
# Obtendo datas de divulgação de resultados da Amazon (AMZN)
ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)
df_earnings = ticker_obj.get_earnings_dates()

In [115]:
df_earnings

,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2026-10-29 16:00:00-04:00,1.95,NaN,NaN
2026-07-30 16:00:00-04:00,1.83,5.75,215.02
2026-04-29 16:00:00-04:00,1.64,2.78,69.02
2026-02-05 16:00:00-05:00,1.95,1.95,0.22
2025-10-30 16:00:00-04:00,1.56,1.95,25.20
2025-07-31 16:00:00-04:00,1.32,1.68,27.19
2025-05-01 16:00:00-04:00,1.36,1.59,16.77
2025-02-06 16:00:00-05:00,1.48,1.86,25.29
2024-10-31 16:00:00-04:00,1.14,1.43,25.22


In [116]:
df_earnings.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 25 entries, 2026-10-29 16:00:00-04:00 to 2020-10-29 16:00:00-04:00
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   EPS Estimate  25 non-null     float64
 1   Reported EPS  24 non-null     float64
 2   Surprise(%)   24 non-null     float64
dtypes: float64(3)
memory usage: 800.0 bytes


In [117]:
# Resetting the index of the earnings DataFrame to make 'Date' a column instead of an index
df_earnings = df_earnings.reset_index()

In [118]:
df_earnings['Earnings Date'] = (
    df_earnings['Earnings Date']
    .dt.tz_localize(None)   # remove o fuso horário
    .dt.normalize()         # zera a hora, deixa só a data (00:00:00)
)

In [119]:
# Merging the earnings DataFrame with the AMZN historical prices DataFrame on the 'Date' column
df_merged = df_amzn.merge(
    df_earnings,
    left_on='Date',
    right_on='Earnings Date',
    how='inner'
)

In [120]:
print(len(df_merged))  # esperado: 24 (os 25 menos o de 2026-10-29, que ainda não aconteceu)

24


In [121]:
# Filtering the merged DataFrame to include only rows where the 'Surprise(%)' column is greater than 0
df_positives = df_merged[df_merged['Surprise(%)'] > 0]


In [122]:
# Calculating the median of the 'retorno_2d' column for rows where 'Surprise(%)' is greater than 0
return_median = df_positives['return_2d'].median()

In [123]:
print(f'Median Return: {return_median}')

Median Return: 0.0035280649406906894


In [124]:
# Calculating the correlation between the 2-day return and the Surprise(%)
correlation = df_merged['return_2d'].corr(df_merged['Surprise(%)'])
print(f'Correlation between 2-day return and Surprise(%): {correlation}')

Correlation between 2-day return and Surprise(%): 0.21906338566924644


In [125]:
df_merged[['return_2d', 'Surprise(%)']].corr()

,return_2d,Surprise(%)
return_2d,1.000000,0.219063
Surprise(%),0.219063,1.000000


In [127]:
from scipy.stats import pearsonr

r, p_value = pearsonr(df_merged['return_2d'], df_merged['Surprise(%)'])
print(f'r = {r:.3f}, p-value = {p_value:.3f}')

r = 0.219, p-value = 0.304


---

### Question 5

In [ ]:
df_stoxx50e_index = get_historical_data(['^STOXX50E'], '2018-01-01', '2026-10-29')
df_stoxx_index = get_historical_data(['^STOXX'], '2018-01-01', '2026-10-29')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [ ]:
df_stoxx_index

,Ticker,Date,Close,High,Low,Open,Volume
0,^STOXX,2018-01-03,390.220001,391.100006,388.739990,388.739990,173214100
1,^STOXX,2018-01-04,393.679993,394.260010,391.109985,391.109985,211746500
2,^STOXX,2018-01-05,397.350006,397.350006,393.980011,393.980011,179155700
3,^STOXX,2018-01-08,398.410004,398.959991,397.720001,397.920013,175894800
4,^STOXX,2018-01-09,400.109985,400.309998,398.489990,398.489990,190142200
...,...,...,...,...,...,...,...
2177,^STOXX,2026-09-04,649.880005,650.909973,647.869995,649.369995,135651500
2178,^STOXX,2026-09-07,649.900024,651.239990,647.400024,649.330017,97779900
2179,^STOXX,2026-09-08,649.599976,650.369995,645.969971,648.960022,125002600
2180,^STOXX,2026-09-09,640.409973,647.900024,638.820007,647.900024,166695300


In [ ]:
df_stoxx50e_index

,Ticker,Date,Close,High,Low,Open,Volume
0,^STOXX50E,2018-01-03,3509.879883,3519.729980,3488.909912,3492.899902,33320900
1,^STOXX50E,2018-01-04,3568.879883,3580.469971,3520.840088,3520.840088,43638600
2,^STOXX50E,2018-01-05,3607.629883,3608.699951,3570.510010,3570.510010,35418700
3,^STOXX50E,2018-01-08,3616.449951,3624.739990,3608.820068,3610.969971,28252700
4,^STOXX50E,2018-01-09,3622.870117,3629.729980,3614.159912,3617.479980,31607400
...,...,...,...,...,...,...,...
2177,^STOXX50E,2026-09-04,6392.930176,6409.379883,6363.350098,6380.700195,13890100
2178,^STOXX50E,2026-09-07,6403.990234,6407.870117,6370.529785,6384.939941,10797200
2179,^STOXX50E,2026-09-08,6413.169922,6416.359863,6362.450195,6394.009766,14230500
2180,^STOXX50E,2026-09-09,6311.560059,6395.009766,6275.339844,6395.009766,21292800


##### Euro Stoxx 50

In [97]:
get_html_tables_name('https://en.wikipedia.org/wiki/EURO_STOXX_50')

16
0 (8, 2) [0, 1]
1 (7, 2) ['Euro Stoxx 50 (DSX)', 'Unnamed: 1']
2 (40, 4) ['Year', 'Year end value', 'Change, in points', 'Change, in\xa0%']
3 (50, 7) ['Ticker', 'Main listing', 'Name', 'Corporate form', 'Registered office']
4 (8, 3) ['Rank', 'Country', 'Stocks']
5 (2, 2) ['vte EURO STOXX 50 companies of the Euro Area', 'vte EURO STOXX 50 companies of the Euro Area.1']
6 (34, 2) ['vteStock market indices', 'vteStock market indices.1']
7 (2, 2) [0, 1]
8 (4, 2) [0, 1]
9 (5, 2) [0, 1]
10 (5, 2) [0, 1]
11 (2, 2) [0, 1]
12 (3, 2) [0, 1]
13 (5, 2) [0, 1]
14 (2, 2) [0, 1]
15 (1, 2) [0, 1]


In [98]:
df_scrapping_euro_stoxx_50 = scrapping('https://en.wikipedia.org/wiki/EURO_STOXX_50', 'Ticker')
df_scrapping_euro_stoxx_50.head(3)

Table Ticker status: True


,Ticker,Main listing,Name,Corporate form,Registered office,Sector,Founded
0,ADS.DE,FWB: ADS,Adidas,Aktiengesellschaft,Germany,Consumer Discretionary,1924
1,ADYEN.AS,Euronext Amsterdam: ADYEN,Adyen,Naamloze vennootschap,Netherlands,Financials,2006
2,AD.AS,Euronext Amsterdam: AD,Ahold Delhaize,Naamloze vennootschap,Netherlands,Consumer Staples,2016 (1867)


In [ ]:
df_euro_stoxx_50, df_empty_tickers =get_historical_data(df_scrapping_euro_stoxx_50, '2018-01-01', '2026-10-29')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [102]:
df_empty_tickers

[]

In [103]:
df_euro_stoxx_50

,Ticker,Date,Close,High,Low,Open,Volume
0,ADS.DE,2018-01-02,152.574478,153.076521,150.840164,152.574478,587067
1,ADS.DE,2018-01-03,153.396027,153.396027,151.570427,152.985269,662050
2,ADS.DE,2018-01-04,155.404175,156.727732,153.806775,154.400089,806719
3,ADS.DE,2018-01-05,157.047226,157.594898,156.088783,156.727740,693116
4,ADS.DE,2018-01-08,157.686157,158.964071,157.366672,158.918442,805948
...,...,...,...,...,...,...,...
109545,WKL.AS,2026-09-03,70.580002,72.040001,68.019997,68.019997,1315841
109546,WKL.AS,2026-09-04,68.860001,71.300003,68.699997,71.260002,1169184
109547,WKL.AS,2026-09-08,67.260002,68.760002,66.800003,68.419998,892551
109548,WKL.AS,2026-09-09,66.800003,67.500000,66.260002,66.800003,1060869


In [91]:
save_dataframe(df_euro_stoxx_50, 'euro_stoxx_50_historical_data')

#### STOXX Europe 600

In [122]:
get_html_tables_name('https://en.wikipedia.org/wiki/STOXX_Europe_600')

15
0 (10, 2) [0, 1]
1 (26, 4) ['Year', 'No. of points', 'Change (points)', 'Change (%)']
2 (1, 2) [0, 1]
3 (467, 5) ['Ticker', 'Company', 'ICB Sector', 'Country', 'Headquarters']
4 (4, 4) ['Date', 'Added', 'Removed', 'Reason']
5 (7, 3) ['Sector', 'Weight (approx.)', 'Key Components']
6 (34, 2) ['vteStock market indices', 'vteStock market indices.1']
7 (2, 2) [0, 1]
8 (4, 2) [0, 1]
9 (5, 2) [0, 1]
10 (5, 2) [0, 1]
11 (2, 2) [0, 1]
12 (3, 2) [0, 1]
13 (5, 2) [0, 1]
14 (2, 2) [0, 1]


In [123]:
df_scrapping_stoxx_europe_600 = scrapping('https://en.wikipedia.org/wiki/STOXX_Europe_600', 'Ticker')
df_scrapping_stoxx_europe_600.head(3)

Table Ticker status: True


,Ticker,Company,ICB Sector,Country,Headquarters
0,ZURN,Zurich Insurance Group,Insurance,Switzerland,Zurich
1,GSK,GSK plc,Health Care,United Kingdom,London
2,RIO,Rio Tinto Group,Basic Resources,United Kingdom,London


In [124]:
df_scrapping_stoxx_europe_600['Ticker'].count()

np.int64(467)

In [125]:
df_scrapping_stoxx_europe_600['Country'].nunique()

20

In [130]:
lista= ['PSN.L', 'PSN.PA']
df_test = get_historical_data_org(lista, '2026-01-01', '2026-10-29')
df_test

TypeError: list indices must be integers or slices, not str

In [126]:
df_stoxx600, df_stoxx600_info = get_eu_historical_data(df_scrapping_stoxx_europe_600, '2018-01-01', '2026-10-29')

$ZURN: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['ZURN']: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$UBSG: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['UBSG']: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$BATS: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-10-29)
[**********

TypeError: cannot use 'pandas.Series' as a dict key (unhashable type: 'Series')

In [127]:
df_scrapping_stoxx_europe_600[df_scrapping_stoxx_europe_600.duplicated(subset='Ticker', keep=False)]

,Ticker,Company,ICB Sector,Country,Headquarters
51,PSN,Persimmon plc,Consumer Products and Services,United Kingdom,York
53,KGF,Kingfisher plc,Retail,United Kingdom,London
66,HELN,Helvetia Insurance,Insurance,Switzerland,St. Gallen
107,BOL,Boliden AB,Basic Resources,Sweden,Stockholm
109,SAN,Banco Santander,Banks,Spain,Madrid
119,BOL,Bolloré,Industrial Goods and Services,France,Puteaux
187,HELN,Helvetia,Insurance,Switzerland,St. Gallen
209,KBC,KBC Group,Banks,Belgium,Brussels
268,PSN,Publicis,Media,France,Paris
283,SAN,Sanofi,Health Care,France,Paris


In [ ]:
df_stoxx600

,Ticker,Date,Close,High,Low,Open,Volume,"(Adj Close, AMBU)","(Close, AMBU)","(Close, B)",...,"(High, TREL)","(Low, TREL)","(Open, TREL)","(Volume, TREL)","(Adj Close, VPLAY)","(Close, VPLAY)","(High, VPLAY)","(Low, VPLAY)","(Open, VPLAY)","(Volume, VPLAY)"
0,GSK,2018-01-02,24.754436,24.794656,24.164566,24.218191,9465500.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GSK,2018-01-03,24.660591,24.834870,24.606965,24.787949,6600800.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GSK,2018-01-04,24.707520,24.787956,24.613677,24.761143,5206400.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GSK,2018-01-05,25.143217,25.143217,24.881798,25.002453,7250700.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GSK,2018-01-08,25.069490,25.196848,24.881803,25.143222,5909100.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292616,NGG,2026-09-03,78.169998,78.169998,77.239998,77.610001,997700.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
292617,NGG,2026-09-04,78.120003,78.599998,77.739998,77.900002,845100.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
292618,NGG,2026-09-08,78.059998,78.269997,77.339996,77.470001,995900.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
292619,NGG,2026-09-09,77.279999,78.680000,77.129997,78.480003,939900.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_stoxx600['Ticker'].unique()

<StringArray>
[ 'GSK',  'RIO', 'INGA',  'SAF',   'DG',  'PRU',  'MBG',  'HLI',  'ISP',
  'BKT',
 ...
 'FLOW', 'FLTR',  'FOR', 'INDV',  'INP',  'IPX', 'JUST', 'LANV',  'MKS',
  'NGG']
Length: 203, dtype: str

In [ ]:
df_stoxx600_info

['ZURN',
 'UBSG',
 'BATS',
 'FERR',
 'RMS',
 'HMB',
 'PERP',
 'MICP',
 'CABK',
 'ENG',
 'NTGY',
 'ITX',
 'ATCOa',
 'ERICb',
 'SAND',
 'FORTUM',
 'GLEN',
 'LLOY',
 'CPG',
 'IMB',
 'LSEG',
 'HLMA',
 'SMT',
 'CRDA',
 'SDLF',
 'BTRW',
 'KGF',
 'HOLN',
 'SRENH',
 'LONN',
 'GIVN',
 'SIKA',
 'STMN',
 'PGHN',
 'SOON',
 'LISN',
 'UHR',
 'HELN',
 'BALN',
 'EMSN',
 'ABBN',
 'ABN',
 'AC',
 'ACS',
 'ADEN',
 'ADS',
 'ADYEN',
 'AENA',
 'AGS',
 'AIRP',
 'AKERBP',
 'ALO',
 'AIXA',
 'ANTO',
 'AKE',
 'ASMI',
 'ABF',
 'AUTO',
 'CS',
 'AZE',
 'BARC',
 'BARN',
 'BAS',
 'BAYN',
 'BEI',
 'BEB',
 'BKG',
 'BNPP',
 'BT.A',
 'BNZL',
 'BRBY',
 'BVI',
 'CAPP',
 'CCH',
 'COLR',
 '1COV',
 'DANO',
 'DHER',
 'DEMANT',
 'DB1',
 'DHL',
 'DPLM',
 'DOCM',
 'SMDS',
 'EDP',
 'EDPR',
 'EFGI',
 'ELUX-B',
 'ELISA',
 'ENI',
 'ENEL',
 'ENT',
 'EOAN',
 'ESLX',
 'ENX',
 'EVK',
 'EXO',
 'FRVIA',
 'FNTN',
 'FME',
 'GALP',
 'GAW',
 'G1A',
 'GEBN',
 'GFC',
 'GET',
 'GN',
 'GFT',
 'HNR',
 'HELN',
 'HEN3',
 'HSX',
 'HWDN',
 'HSBA',
 'INC

In [ ]:
count = 0
for i in range(len(df_stoxx600_info)):
    count += 1
print(f"Total tickers processed: {count}")

Total tickers processed: 259
